# 03 – Construct: Controllable benchmark and delay propagation
Goal: build the final metrics that answer the client question — where each airline's controllable disruption exceeds its peers — and test whether delays propagate through aircraft rotation.

In [1]:
import glob
import pandas as pd

clientes = ["AA", "DL", "UA", "B6", "WN"]
causas = ["CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay"]
cols = ["Year", "Month", "FlightDate", "Reporting_Airline", "Tail_Number",
        "Flight_Number_Reporting_Airline", "Origin", "Dest", "CRSDepTime", "DepTime",
        "DepDelay", "ArrDelay", "ArrDel15", "Cancelled", "Diverted", "CancellationCode"] + causas

def leer(archivo):
    df = pd.read_parquet(archivo, columns=cols, filters=[("Reporting_Airline", "in", clientes)])
    for c in ["Reporting_Airline", "Tail_Number", "Origin", "Dest", "CancellationCode", "CRSDepTime"]:
        df[c] = df[c].astype("category")
    df["FlightDate"] = pd.to_datetime(df["FlightDate"])
    return df

archivos = sorted(glob.glob("data/interim/ontime_*.parquet"))
anio = pd.concat([leer(f) for f in archivos], ignore_index=True)

for c in ["Reporting_Airline", "Tail_Number", "Origin", "Dest", "CancellationCode", "CRSDepTime"]:
    anio[c] = anio[c].astype("category")

print(f"Archivos leídos: {len(archivos)}")
print(f"Vuelos cargados: {len(anio):,}")
print(f"Memoria usada:   {anio.memory_usage(deep=True).sum() / 1e9:.2f} GB")

Archivos leídos: 12
Vuelos cargados: 4,471,010
Memoria usada:   0.98 GB


In [2]:
principales = {aer: anio.loc[anio["Reporting_Airline"] == aer, "Origin"].value_counts().idxmax()
               for aer in clientes}
print("Aeropuerto principal de cada cliente:", principales)

def metricas(g):
    op = g[(g["Cancelled"] == 0) & (g["Diverted"] == 0)]
    return {
        "programados": len(g),
        "operados": len(op),
        "ctrl_pct": ((op["ArrDel15"] == 1) & (op["CarrierDelay"] > 0)).mean() * 100,
        "min_carrier": op["CarrierDelay"].sum() / len(op),
        "cancel_A": (g["CancellationCode"] == "A").sum() / len(g) * 100,
    }

filas = []
for apt in sorted(set(principales.values())):
    en_apt = anio[anio["Origin"] == apt]
    for aer in clientes:
        propia = en_apt[en_apt["Reporting_Airline"] == aer]
        if len(propia) < 100:
            continue
        pares = en_apt[en_apt["Reporting_Airline"] != aer]
        m, p = metricas(propia), metricas(pares)
        filas.append({
            "aeropuerto": apt,
            "aerolinea": aer,
            "vuelos": m["programados"],
            "ctrl_pct": round(m["ctrl_pct"], 1),
            "ctrl_pct_pares": round(p["ctrl_pct"], 1),
            "brecha_ctrl": round(m["ctrl_pct"] - p["ctrl_pct"], 1),
            "min_carrier": round(m["min_carrier"], 1),
            "min_carrier_pares": round(p["min_carrier"], 1),
            "cancel_A": round(m["cancel_A"], 2),
            "cancel_A_pares": round(p["cancel_A"], 2),
            "exceso_ctrl": round(m["operados"] * (m["ctrl_pct"] - p["ctrl_pct"]) / 100),
        })

bench = pd.DataFrame(filas)
bench.sort_values("exceso_ctrl", ascending=False)

Aeropuerto principal de cada cliente: {'AA': 'DFW', 'DL': 'ATL', 'UA': 'DEN', 'B6': 'BOS', 'WN': 'DEN'}


,aeropuerto,aerolinea,vuelos,ctrl_pct,ctrl_pct_pares,brecha_ctrl,min_carrier,min_carrier_pares,cancel_A,cancel_A_pares,exceso_ctrl
15,DFW,AA,164581,25.2,8.8,16.4,8.1,5.8,0.25,0.83,26136
14,DEN,WN,88768,20.5,12.0,8.6,5.3,4.3,0.14,0.20,7542
8,BOS,B6,37487,17.1,12.3,4.8,5.1,5.3,0.38,0.48,1736
9,BOS,WN,7197,21.0,13.5,7.5,4.0,5.3,0.13,0.46,530
4,ATL,WN,17795,15.0,13.1,1.9,3.6,5.7,0.17,0.94,340
3,ATL,B6,2575,16.5,13.2,3.4,8.0,5.5,0.54,0.89,84
13,DEN,B6,1249,16.7,15.5,1.2,10.0,4.7,0.56,0.17,15
18,DFW,B6,364,14.8,23.3,-8.5,14.0,7.8,0.55,0.32,-30
0,ATL,AA,6810,12.5,13.2,-0.7,7.8,5.5,0.18,0.90,-44
10,DEN,AA,8874,13.8,15.5,-1.7,9.5,4.5,0.28,0.17,-147


In [3]:
principales = {aer: anio.loc[anio["Reporting_Airline"] == aer, "Origin"].value_counts().idxmax()
               for aer in clientes}
print("Aeropuerto principal de cada cliente:", principales)

def metricas(g):
    op = g[(g["Cancelled"] == 0) & (g["Diverted"] == 0)]
    return {
        "programados": len(g),
        "operados": len(op),
        "ctrl_pct": ((op["ArrDel15"] == 1) & (op["CarrierDelay"] > 0)).mean() * 100,
        "min_carrier": op["CarrierDelay"].sum() / len(op),
        "cancel_A": (g["CancellationCode"] == "A").sum() / len(g) * 100,
    }

filas = []
for apt in sorted(set(principales.values())):
    en_apt = anio[anio["Origin"] == apt]
    for aer in clientes:
        propia = en_apt[en_apt["Reporting_Airline"] == aer]
        if len(propia) < 100:
            continue
        pares = en_apt[en_apt["Reporting_Airline"] != aer]
        m, p = metricas(propia), metricas(pares)
        filas.append({
            "aeropuerto": apt,
            "aerolinea": aer,
            "programados": m["programados"],
            "operados": m["operados"],
            "ctrl_pct": round(m["ctrl_pct"], 1),
            "ctrl_pct_pares": round(p["ctrl_pct"], 1),
            "brecha_ctrl": round(m["ctrl_pct"] - p["ctrl_pct"], 1),
            "min_carrier": round(m["min_carrier"], 1),
            "min_carrier_pares": round(p["min_carrier"], 1),
            "cancel_A": round(m["cancel_A"], 2),
            "cancel_A_pares": round(p["cancel_A"], 2),
            "exceso_ctrl": round(m["operados"] * (m["ctrl_pct"] - p["ctrl_pct"]) / 100),
        })

bench = pd.DataFrame(filas)
bench.sort_values("exceso_ctrl", ascending=False)

Aeropuerto principal de cada cliente: {'AA': 'DFW', 'DL': 'ATL', 'UA': 'DEN', 'B6': 'BOS', 'WN': 'DEN'}


,aeropuerto,aerolinea,programados,operados,ctrl_pct,ctrl_pct_pares,brecha_ctrl,min_carrier,min_carrier_pares,cancel_A,cancel_A_pares,exceso_ctrl
15,DFW,AA,164581,158920,25.2,8.8,16.4,8.1,5.8,0.25,0.83,26136
14,DEN,WN,88768,87968,20.5,12.0,8.6,5.3,4.3,0.14,0.20,7542
8,BOS,B6,37487,36425,17.1,12.3,4.8,5.1,5.3,0.38,0.48,1736
9,BOS,WN,7197,7070,21.0,13.5,7.5,4.0,5.3,0.13,0.46,530
4,ATL,WN,17795,17530,15.0,13.1,1.9,3.6,5.7,0.17,0.94,340
3,ATL,B6,2575,2487,16.5,13.2,3.4,8.0,5.5,0.54,0.89,84
13,DEN,B6,1249,1210,16.7,15.5,1.2,10.0,4.7,0.56,0.17,15
18,DFW,B6,364,352,14.8,23.3,-8.5,14.0,7.8,0.55,0.32,-30
0,ATL,AA,6810,6643,12.5,13.2,-0.7,7.8,5.5,0.18,0.90,-44
10,DEN,AA,8874,8659,13.8,15.5,-1.7,9.5,4.5,0.28,0.17,-147


In [4]:
import glob
import pandas as pd

clientes = ["AA", "DL", "UA", "B6", "WN"]
causas = ["CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay"]
cols = ["Year", "Month", "FlightDate", "Reporting_Airline", "Tail_Number",
        "Flight_Number_Reporting_Airline", "Origin", "Dest", "CRSDepTime", "CRSArrTime", "DepTime",
        "DepDelay", "ArrDelay", "ArrDel15", "Cancelled", "Diverted", "CancellationCode"] + causas
texto_cat = ["Reporting_Airline", "Tail_Number", "Origin", "Dest", "CancellationCode", "CRSDepTime", "CRSArrTime"]

def leer(archivo):
    df = pd.read_parquet(archivo, columns=cols, filters=[("Reporting_Airline", "in", clientes)])
    for c in texto_cat:
        df[c] = df[c].astype("category")
    df["FlightDate"] = pd.to_datetime(df["FlightDate"])
    return df

archivos = sorted(glob.glob("data/interim/ontime_*.parquet"))
anio = pd.concat([leer(f) for f in archivos], ignore_index=True)
for c in texto_cat:
    anio[c] = anio[c].astype("category")

print(f"Archivos leídos: {len(archivos)}")
print(f"Vuelos cargados: {len(anio):,}")
print(f"Memoria usada:   {anio.memory_usage(deep=True).sum() / 1e9:.2f} GB")

Archivos leídos: 12
Vuelos cargados: 4,471,010
Memoria usada:   0.99 GB


In [5]:
def a_minutos(s):
    x = s.astype(str).astype(int)
    return (x // 100) * 60 + x % 100

op = anio[(anio["Cancelled"] == 0) & (anio["Diverted"] == 0) & anio["Tail_Number"].notna()].copy()
op["dep_min"] = a_minutos(op["CRSDepTime"])
op["arr_min"] = a_minutos(op["CRSArrTime"])
op = op.sort_values(["Tail_Number", "FlightDate", "dep_min"])

jornada = op.groupby(["Tail_Number", "FlightDate"], observed=True)
op["tramos_dia"] = jornada["dep_min"].transform("size")
op["previo_destino"] = jornada["Dest"].shift(1)
op["previo_retraso_llegada"] = jornada["ArrDelay"].shift(1)
op["previo_llegada_prog"] = jornada["arr_min"].shift(1)
op["tierra_prog"] = op["dep_min"] - op["previo_llegada_prog"]

cadena = op[op["previo_retraso_llegada"].notna()
            & (op["previo_destino"].astype(str) == op["Origin"].astype(str))
            & op["tierra_prog"].between(1, 360)].copy()
cadena["en_principal"] = cadena["Origin"].astype(str) == cadena["Reporting_Airline"].astype(str).map(principales)

hub = cadena[cadena["en_principal"]]
recibe_tarde = hub[hub["previo_retraso_llegada"] >= 15]
por_aer = lambda df: df.groupby("Reporting_Airline", observed=True)

rec = pd.DataFrame({
    "pares_en_hub": por_aer(hub).size(),
    "tierra_prog_mediana": por_aer(hub)["tierra_prog"].median(),
    "pct_recibe_tarde": por_aer(hub)["previo_retraso_llegada"].apply(lambda s: (s >= 15).mean() * 100),
    "llega_con_min": por_aer(recibe_tarde)["previo_retraso_llegada"].mean(),
    "sale_con_min": por_aer(recibe_tarde)["DepDelay"].mean(),
    "pct_sale_a_tiempo": por_aer(recibe_tarde)["DepDelay"].apply(lambda s: (s < 15).mean() * 100),
})
rec["recupera_en_tierra"] = rec["llega_con_min"] - rec["sale_con_min"]
dias_avion = op.drop_duplicates(["Tail_Number", "FlightDate"])
rec["tramos_por_dia"] = por_aer(dias_avion)["tramos_dia"].mean()
rec = rec.round(1)
rec.index = [f"{a} en {principales[a]}" for a in rec.index]

print(f"Pares de vuelos consecutivos válidos: {len(cadena):,}")
rec


Pares de vuelos consecutivos válidos: 3,044,604


,pares_en_hub,tierra_prog_mediana,pct_recibe_tarde,llega_con_min,sale_con_min,pct_sale_a_tiempo,recupera_en_tierra,tramos_por_dia
AA en DFW,111146,66.0,21.9,65.9,49.2,29.8,16.7,3.4
B6 en BOS,20680,63.0,25.1,64.5,58.2,21.7,6.4,2.9
DL en ATL,186536,70.0,12.4,60.5,40.4,36.7,20.1,3.7
UA en DEN,82053,74.0,16.3,59.4,38.7,43.3,20.8,3.0
WN en DEN,76273,50.0,23.9,51.8,51.9,13.3,-0.0,5.0


### Construct Finding 2 — Recovery from late-arriving aircraft (12 months)
3,044,604 valid consecutive flight pairs (same aircraft, same day, arrival airport = next departure airport, 1–360 min scheduled ground time). Shown: each airline at its main airport.

| Airline | Scheduled ground time (median) | Minutes recovered on ground | Departs on time after receiving a late aircraft |
|---|---|---|---|
| WN at DEN | 50 | 0.0 | 13.3% |
| B6 at BOS | 63 | 6.4 | 21.7% |
| AA at DFW | 66 | 16.7 | 29.8% |
| DL at ATL | 70 | 20.1 | 36.7% |
| UA at DEN | 74 | 20.8 | 43.3% |

- **H3 (schedule buffer) supported:** recovery ranks in the same order as scheduled ground time across all 5 airlines. Strong, consistent relationship; not proof of causation.
- **Denver explained:** same airport — UA (74 min) recovers 20.8 min; WN (50 min) recovers none.
- **H4 (aircraft utilization) not supported as a general rule:** DL flies more legs per aircraft-day than AA (3.7 vs. 3.4) yet recovers more.
- **AA at DFW:** receives late aircraft almost twice as often as DL at ATL (21.9% vs. 12.4%) and recovers less with a similar buffer → points to ground execution, consistent with Construct Finding 1.
- ⚠️ Staffing and maintenance cannot be separated with BTS data; they are grouped inside CarrierDelay.

In [6]:
bins = [0, 9, 12, 15, 18, 21, 24]
etiquetas = ["antes 09", "09-11", "12-14", "15-17", "18-20", "21+"]

def franja(df):
    return pd.cut(df["dep_min"] // 60, bins=bins, labels=etiquetas, right=False)

def nombrar(tabla):
    tabla.columns = [f"{a} en {principales[a]}" for a in tabla.columns]
    return tabla

# Tabla 1: % de salidas tarde por franja (todas las salidas desde el aeropuerto principal)
op_hub = op[op["Origin"].astype(str) == op["Reporting_Airline"].astype(str).map(principales)].copy()
op_hub["franja"] = franja(op_hub)
op_hub["sale_tarde"] = op_hub["DepDelay"] >= 15
tabla1 = nombrar((op_hub.pivot_table(index="franja", columns="Reporting_Airline", values="sale_tarde",
                                     aggfunc="mean", observed=True) * 100).round(1))

# Tabla 2: minutos recuperados en tierra cuando el avión llegó tarde
rt = recibe_tarde.copy()
rt["franja"] = franja(rt)
rt["recupera"] = rt["previo_retraso_llegada"] - rt["DepDelay"]
tabla2 = nombrar(rt.pivot_table(index="franja", columns="Reporting_Airline", values="recupera",
                                aggfunc="mean", observed=True).round(1))

# Tabla 3: tiempo programado en tierra (mediana) por franja
h = hub.copy()
h["franja"] = franja(h)
tabla3 = nombrar(h.pivot_table(index="franja", columns="Reporting_Airline", values="tierra_prog",
                               aggfunc="median", observed=True))

print("TABLA 1 — % de salidas tarde (15+ min)")
display(tabla1)
print("TABLA 2 — Minutos recuperados en tierra (cuando el avión llegó tarde)")
display(tabla2)
print("TABLA 3 — Minutos programados en tierra (mediana)")
display(tabla3)

TABLA 1 — % de salidas tarde (15+ min)


,AA en DFW,B6 en BOS,DL en ATL,UA en DEN,WN en DEN
franja,,,,,
antes 09,19.9,10.8,9.0,6.8,9.9
09-11,29.3,18.9,13.5,14.0,23.1
12-14,36.3,27.6,17.8,17.1,35.0
15-17,42.4,38.8,23.8,23.6,47.3
18-20,48.3,45.0,26.4,30.4,57.0
21+,48.0,48.0,22.0,22.5,52.5


TABLA 2 — Minutos recuperados en tierra (cuando el avión llegó tarde)


,AA en DFW,B6 en BOS,DL en ATL,UA en DEN,WN en DEN
franja,,,,,
antes 09,14.0,1.9,19.0,17.3,4.1
09-11,16.9,3.3,20.4,19.5,7.5
12-14,17.8,4.8,31.4,40.1,-1.3
15-17,20.6,6.4,13.1,15.6,-2.8
18-20,13.4,7.8,18.7,16.9,-1.1
21+,17.0,5.6,21.4,38.7,4.4


TABLA 3 — Minutos programados en tierra (mediana)


,AA en DFW,B6 en BOS,DL en ATL,UA en DEN,WN en DEN
franja,,,,,
antes 09,65.0,55.0,66.0,71.0,55.0
09-11,66.0,57.0,69.0,73.0,50.0
12-14,64.0,60.0,66.0,82.0,45.0
15-17,66.0,67.0,67.0,73.0,45.0
18-20,66.0,70.0,74.0,73.0,50.0
21+,71.0,66.0,77.0,97.0,55.0


In [7]:
temprano = op_hub[op_hub["franja"] == "antes 09"]
primeros = temprano[temprano["previo_destino"].isna()]
por_aer = lambda df: df.groupby("Reporting_Airline", observed=True)

verif = pd.DataFrame({
    "salidas_antes_09": por_aer(temprano).size(),
    "pct_primer_vuelo_del_dia": (por_aer(temprano)["previo_destino"].apply(lambda s: s.isna().mean() * 100)).round(1),
    "pct_tarde_primer_vuelo": (por_aer(primeros)["sale_tarde"].mean() * 100).round(1),
})
verif.index = [f"{a} en {principales[a]}" for a in verif.index]
verif

,salidas_antes_09,pct_primer_vuelo_del_dia,pct_tarde_primer_vuelo
AA en DFW,29664,64.0,18.0
B6 en BOS,10326,89.0,9.7
DL en ATL,29670,61.4,9.0
UA en DEN,10118,88.4,5.1
WN en DEN,16901,43.3,6.2


### Construct Finding 3 — Time of day and first-flight readiness (12 months)
Each airline at its main airport; 3-hour bands by scheduled local departure time.

**Delay accumulates through the day for every airline**, but the pattern differs:
- **WN at DEN:** steepest rise (9.9% → 57.0% of departures late). It schedules its **shortest ground times (45 min) in the 12:00–17:59 bands**, where its recovery turns **negative** (the aircraft leaves later than it arrived).
- **UA at DEN (same airport):** widens ground time to 82 min at midday and 97 min at night, recovering about 39–40 min in those bands — the best-practice benchmark.
- **AA at DFW:** flat ground-time schedule all day (64–66 min, 71 after 21:00); it does not adapt to afternoon accumulation.
- **B6 at BOS:** low recovery in every band (1.9–7.8 min) despite buffers similar to DL's in the afternoon → ground execution.

**First flight of the day (no inbound aircraft, so no propagation possible):**

| Airline | First flights departing late |
|---|---|
| AA at DFW | **18.0%** |
| B6 at BOS | 9.7% |
| DL at ATL | 9.0% |
| WN at DEN | 6.2% |
| UA at DEN | 5.1% |

- AA's morning problem is **station readiness**, the purest controllable signal: about 2x DL and 3.5x UA.
- Verification note: only 43–89% of pre-09:00 departures are first flights, so the first-flight rate (not the pre-09:00 rate) is the metric reported. "First flight" = first **operated** flight of the aircraft that day.
- ⚠️ Within each airline, ground time and recovery do not align perfectly band by band; the strong relationship is **across** airlines.

In [8]:
import os
os.makedirs("results", exist_ok=True)

def separar(df, col="case"):
    df["airline"] = df[col].str[:2]
    df["airport"] = df[col].str[-3:]
    return df.drop(columns=col)

# 1. Benchmark controlable (12 meses)
b = bench.rename(columns={
    "aeropuerto": "airport", "aerolinea": "airline",
    "programados": "scheduled_flights", "operados": "operated_flights",
    "ctrl_pct": "ctrl_late_pct", "ctrl_pct_pares": "peer_ctrl_late_pct", "brecha_ctrl": "ctrl_gap_pts",
    "min_carrier": "carrier_min_per_flight", "min_carrier_pares": "peer_carrier_min_per_flight",
    "cancel_A": "carrier_cancel_pct", "cancel_A_pares": "peer_carrier_cancel_pct",
    "exceso_ctrl": "excess_ctrl_late_flights"})
b["is_main_airport"] = b["airport"] == b["airline"].map(principales)
b.to_csv("results/benchmark_controllable.csv", index=False)

# 2. Brecha controlable mes a mes (cada aerolínea en su aeropuerto principal)
filas = []
for aer in clientes:
    apt = principales[aer]
    en_apt = anio[anio["Origin"] == apt]
    for (y, mes), g in en_apt.groupby(["Year", "Month"]):
        m = metricas(g[g["Reporting_Airline"] == aer])
        p = metricas(g[g["Reporting_Airline"] != aer])
        filas.append({"airline": aer, "airport": apt, "month": f"{y}-{mes:02d}",
                      "ctrl_late_pct": round(m["ctrl_pct"], 1),
                      "peer_ctrl_late_pct": round(p["ctrl_pct"], 1),
                      "ctrl_gap_pts": round(m["ctrl_pct"] - p["ctrl_pct"], 1)})
mensual = pd.DataFrame(filas)
mensual.to_csv("results/monthly_controllable_gap.csv", index=False)

# 3. Recuperación por aerolínea
r = separar(rec.reset_index().rename(columns={"index": "case"})).rename(columns={
    "pares_en_hub": "consecutive_pairs", "tierra_prog_mediana": "ground_time_median_min",
    "pct_recibe_tarde": "pct_inbound_late", "llega_con_min": "inbound_delay_min",
    "sale_con_min": "outbound_delay_min", "pct_sale_a_tiempo": "pct_departs_on_time_after_late_inbound",
    "recupera_en_tierra": "recovery_min", "tramos_por_dia": "legs_per_aircraft_day"})
r.to_csv("results/recovery_by_airline.csv", index=False)

# 4. Hora del día (formato largo)
def largo(t, nombre):
    return t.reset_index().melt(id_vars="franja", var_name="case", value_name=nombre)
tod = (largo(tabla1, "pct_departing_late")
       .merge(largo(tabla2, "recovery_min"), on=["franja", "case"])
       .merge(largo(tabla3, "ground_time_median_min"), on=["franja", "case"]))
tod["time_band"] = tod["franja"].astype(str).replace({"antes 09": "before 09"})
tod = separar(tod.drop(columns="franja"))
tod.to_csv("results/time_of_day.csv", index=False)

# 5. Primer vuelo del día
f = separar(verif.reset_index().rename(columns={"index": "case"})).rename(columns={
    "salidas_antes_09": "departures_before_09", "pct_primer_vuelo_del_dia": "pct_first_flight_of_day",
    "pct_tarde_primer_vuelo": "pct_first_flights_late"})
f.to_csv("results/first_flight.csv", index=False)

for nombre, df in [("benchmark_controllable", b), ("monthly_controllable_gap", mensual),
                   ("recovery_by_airline", r), ("time_of_day", tod), ("first_flight", f)]:
    print(f"results/{nombre}.csv  →  {len(df)} filas, {len(df.columns)} columnas")

mensual.pivot(index="month", columns="airline", values="ctrl_gap_pts")

results/benchmark_controllable.csv  →  19 filas, 13 columnas
results/monthly_controllable_gap.csv  →  60 filas, 6 columnas
results/recovery_by_airline.csv  →  5 filas, 10 columnas
results/time_of_day.csv  →  30 filas, 6 columnas
results/first_flight.csv  →  5 filas, 5 columnas


airline,AA,B6,DL,UA,WN
month,,,,,
2025-07,21.6,7.3,1.1,-2.0,5.7
2025-08,21.2,3.4,-0.4,-2.9,5.7
2025-09,13.5,-3.0,0.3,-5.1,7.6
2025-10,15.2,-0.1,-2.4,-9.4,11.8
2025-11,13.2,5.9,-0.1,-3.9,5.2
2025-12,18.3,11.5,2.2,-6.4,7.2
2026-01,16.6,9.2,5.1,-1.8,2.2
2026-02,14.2,11.2,2.8,-4.1,5.3
2026-03,14.4,6.6,1.8,-6.8,8.6


### Construct Finding 4 — Monthly controllable gap and dashboard exports
Controllable late-rate gap vs. same-airport peers, each airline at its main airport, month by month.

| Airline | Months worse than peers | Avg. monthly gap | Range | Pattern |
|---|---|---|---|---|
| AA at DFW | 12 / 12 | +16.4 pts | +12.8 to +21.6 | Structural — never below +12.8 |
| WN at DEN | 12 / 12 | +8.4 pts | +2.2 to +16.5 | Worsening: Apr–Jun 2026 are its 3 worst months |
| B6 at BOS | 10 / 12 | +5.0 pts | −3.0 to +11.5 | Seasonal (winter) |
| DL at ATL | 7 / 12 | −0.1 pts | −7.1 to +5.1 | Neutral |
| UA at DEN | 0 / 12 | −6.2 pts | −12.9 to −1.8 | Better than peers every month |

Reconciliation: AA's average monthly gap (+16.4) matches its 12-month pooled gap. Small differences for other airlines reflect simple vs. flight-weighted averaging.

**Exports for the dashboard (`results/`):** benchmark_controllable (19 rows), monthly_controllable_gap (60), recovery_by_airline (5), time_of_day (30), first_flight (5).